In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

##Define tables

In [0]:
BRONZE_TABLE = "workspace.bronze.transactions"

SILVER_TABLE = "workspace.silver.transactions"

QUARANTINE_TABLE = "workspace.silver.transaction_quarantine"

print("Bronze:", BRONZE_TABLE)
print("Silver:", SILVER_TABLE)
print("Quarantine:", QUARANTINE_TABLE)

Bronze: workspace.bronze.transactions
Silver: workspace.silver.transactions
Quarantine: workspace.silver.transaction_quarantine


In [0]:
bronze_df = spark.table(BRONZE_TABLE)

print(
    "Bronze records:",
    f"{bronze_df.count():,}"
)

Bronze records: 5,010,000


## Deduplication

In [0]:
dedup_window = (
    Window
    .partitionBy("txn_id")
    .orderBy(F.col("_ingest_ts").desc())
)

deduped_df = (
    bronze_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(
    "After deduplication:",
    f"{deduped_df.count():,}"
)

After deduplication: 5,000,000


###Validation rules

In [0]:
validated_df = (
    deduped_df
    .withColumn(
        "_validation_reason",
        F.when(
            F.col("amount").isNull(),
            "NULL_AMOUNT"
        )
        .when(
            F.col("amount") <= 0,
            "INVALID_AMOUNT"
        )
        .when(
            ~F.col("currency").isin("INR", "USD", "EUR", "GBP"),
            "INVALID_CURRENCY"
        )
        .when(
            F.col("account_id").startswith("INVALID_ACC_"),
            "INVALID_ACCOUNT"
        )
        .when(
            F.col("customer_id").startswith("INVALID_CUST_"),
            "INVALID_CUSTOMER"
        )
        .when(
            ~F.col("txn_type").isin("DEBIT", "CREDIT"),
            "INVALID_TXN_TYPE"
        )
        .when(
            F.col("status") != "SUCCESS",
            "UNSUCCESSFUL_TRANSACTION"
        )
        .otherwise(None)
    )
)

##Validation results

In [0]:
display(
    validated_df
    .groupBy("_validation_reason")
    .count()
    .orderBy(F.desc("count"))
)

_validation_reason,count
null,4681541
UNSUCCESSFUL_TRANSACTION,298549
INVALID_AMOUNT,4992
INVALID_CUSTOMER,4983
INVALID_CURRENCY,4983
INVALID_ACCOUNT,4952


##Create quarantine dataset

In [0]:
quarantine_df = (
    validated_df
    .filter(F.col("_validation_reason").isNotNull())
    .withColumn(
        "_quarantine_ts",
        F.current_timestamp()
    )
)

In [0]:
print(
    "Quarantined records:",
    f"{quarantine_df.count():,}"
)

Quarantined records: 318,459


##Create clean Silver dataset

In [0]:
silver_df = (
    validated_df
    .filter(F.col("_validation_reason").isNull())
    .drop("_validation_reason")
)

In [0]:
print(
    "Valid Silver records:",
    f"{silver_df.count():,}"
)

Valid Silver records: 4,681,541


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

##Quarantine table

In [0]:
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Quarantine table created")

Quarantine table created


##Silver table

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver transaction table created")

Silver transaction table created


#Final Silver validation

In [0]:
silver_count = spark.table(SILVER_TABLE).count()

quarantine_count = spark.table(
    QUARANTINE_TABLE
).count()

print("Silver records:", f"{silver_count:,}")
print("Quarantine records:", f"{quarantine_count:,}")
print(
    "Total:",
    f"{silver_count + quarantine_count:,}"
)

Silver records: 4,681,541
Quarantine records: 318,459
Total: 5,000,000


In [0]:
silver_duplicates = (
    spark.table(SILVER_TABLE)
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate transaction IDs in Silver:",
    silver_duplicates
)

Duplicate transaction IDs in Silver: 0


In [0]:
invalid_silver_amounts = (
    spark.table(SILVER_TABLE)
    .filter(F.col("amount") <= 0)
    .count()
)

print(
    "Invalid amounts in Silver:",
    invalid_silver_amounts
)

Invalid amounts in Silver: 0


In [0]:
invalid_silver_currency = (
    spark.table(SILVER_TABLE)
    .filter(
        ~F.col("currency").isin(
            "INR", "USD", "EUR", "GBP"
        )
    )
    .count()
)

print(
    "Invalid currencies in Silver:",
    invalid_silver_currency
)

Invalid currencies in Silver: 0


In [0]:
invalid_accounts_silver = (
    spark.table(SILVER_TABLE)
    .filter(
        F.col("account_id").startswith("INVALID_ACC_")
    )
    .count()
)

invalid_customers_silver = (
    spark.table(SILVER_TABLE)
    .filter(
        F.col("customer_id").startswith("INVALID_CUST_")
    )
    .count()
)

print("Invalid accounts:", invalid_accounts_silver)
print("Invalid customers:", invalid_customers_silver)

Invalid accounts: 0
Invalid customers: 0


In [0]:
successful_check = (
    spark.table(SILVER_TABLE)
    .filter(F.col("status") != "SUCCESS")
    .count()
)

print(
    "Non-success transactions in Silver:",
    successful_check
)

Non-success transactions in Silver: 0
